In [16]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

load_dotenv()

os.environ['HADOOP_HOME'] = "C:\\hadoop"

spark = SparkSession.builder \
    .appName("PagilaSparkTask") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .config("spark.driver.host", "localhost") \
    .getOrCreate()

db_url = os.getenv("DB_URL")
db_properties = {
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "driver": "org.postgresql.Driver"
}

def load_table(table_name):
    return spark.read.jdbc(db_url, table_name, properties=db_properties)

actor = load_table("actor")
category = load_table("category")
film = load_table("film")
film_actor = load_table("film_actor")
film_category = load_table("film_category")
inventory = load_table("inventory")
rental = load_table("rental")
payment = load_table("payment")
customer = load_table("customer")
address = load_table("address")
city = load_table("city")

print("Все таблицы успешно загружены в PySpark DataFrames!")


Все таблицы успешно загружены в PySpark DataFrames!


In [9]:
task1 = (
    film_category.join(category, "category_id")
    .groupBy("name")
    .agg(F.count("film_id").alias("movie_count"))
    .orderBy(F.desc("movie_count"))
)

print('Запрос 1: Фильмы по категориям')
task1.show(10)



Запрос 1: Фильмы по категориям
+---------+-----------+
|     name|movie_count|
+---------+-----------+
|    Drama|        152|
|    Music|        152|
|   Travel|        151|
|  Foreign|        150|
|    Games|        150|
| Children|        150|
|   Action|        149|
|   Sci-Fi|        149|
|Animation|        148|
|   Family|        147|
+---------+-----------+
only showing top 10 rows


In [10]:
task2 = (
    rental
    .join(inventory, "inventory_id")
    .join(film_actor, "film_id")
    .join(actor, "actor_id")
    .groupBy("actor_id", "first_name", "last_name")
    .agg(F.count("rental_id").alias("rental_count"))
    .orderBy(F.desc("rental_count"))
    .limit(10)
    
)

print('Запрос 2: Топ 10 актеров по арендам')
task2.show(10)

Запрос 2: Топ 10 актеров по арендам
+--------+----------+-----------+------------+
|actor_id|first_name|  last_name|rental_count|
+--------+----------+-----------+------------+
|     107|      GINA|  DEGENERES|        2426|
|     181|   MATTHEW|     CARREY|        2247|
|     198|      MARY|     KEITEL|        2173|
|     144|    ANGELA|WITHERSPOON|        2146|
|     102|    WALTER|       TORN|        2074|
|      37|       VAL|     BOLGER|        2012|
|     150|     JAYNE|      NOLTE|        2010|
|      60|     HENRY|      BERRY|        1978|
|      23|    SANDRA|     KILMER|        1958|
|      90|      SEAN|    GUINESS|        1903|
+--------+----------+-----------+------------+



In [11]:
task3 = (
    payment
    .join(rental, "rental_id")
    .join(inventory, "inventory_id")
    .join(film_category, "film_id")
    .join(category, "category_id")
    .groupBy("category_id", "name")
    .agg(F.sum("amount").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(1)
)

print('Запрос 3: Самая дорогая категория')
task3.show(1)

Запрос 3: Самая дорогая категория
+-----------+------+-------------+
|category_id|  name|total_revenue|
+-----------+------+-------------+
|          1|Action|     26505.44|
+-----------+------+-------------+



In [12]:
task4 = (
    film
    .join(inventory, "film_id", "left")
    .filter(F.col("inventory_id").isNull())
    .select("title")
)

print('Запрос 4: Фильмы, которых нет в каталоге')
task4.show(10)

Запрос 4: Фильмы, которых нет в каталоге
+--------------------+
|               title|
+--------------------+
|      CHOCOLATE DUCK|
|       BUTCH PANTHER|
|        VOLUME HOUSE|
|      ORDER BETRAYED|
|        TADPOLE PARK|
|    KILL BROTHERHOOD|
|FRANKENSTEIN STRA...|
|    CROSSING DIVORCE|
|    SUICIDES SILENCE|
|       CATCH AMISTAD|
+--------------------+
only showing top 10 rows


In [13]:
window_spec = Window.orderBy(F.desc("film_count"))

children_actors = (
    category
    .filter(F.col("name") == "Children")   
    .join(film_category, "category_id")
    .join(film_actor, "film_id")
    .join(actor, "actor_id")
    .groupBy("actor_id", "first_name", "last_name")  
    .agg(F.count("film_id").alias("film_count"))    
)

task5 = (
    children_actors
    .withColumn("rnk", F.dense_rank().over(window_spec))  
    .filter(F.col("rnk") <= 3)                             
    .drop("rnk")                                          
)

print('Запрос 5:  Актеры, которые чаще всего снимались в фильмах категории «Детские»')
task5.show()

Запрос 5:  Актеры, которые чаще всего снимались в фильмах категории «Детские»
+--------+----------+---------+----------+
|actor_id|first_name|last_name|film_count|
+--------+----------+---------+----------+
|     105|    SIDNEY|    CROWE|         9|
|     139|      EWAN|  GOODING|         9|
|     133|   RICHARD|     PENN|         9|
|      87|   SPENCER|     PECK|         8|
|     145|       KIM|    ALLEN|         8|
|      66|      MARY|    TANDY|         8|
|      29|      ALEC|    WAYNE|         8|
|      56|       DAN|   HARRIS|         8|
|     149|   RUSSELL|   TEMPLE|         8|
|     181|   MATTHEW|   CARREY|         8|
|     131|      JANE|  JACKMAN|         8|
|     142|      JADA|    RYDER|         8|
|      84|     JAMES|     PITT|         7|
|     108|    WARREN|    NOLTE|         7|
|     123|  JULIANNE|    DENCH|         7|
|      34|    AUDREY|  OLIVIER|         7|
|      96|      GENE|   WILLIS|         7|
|      65|    ANGELA|   HUDSON|         7|
|      95|     DARY

In [14]:
task6 = (
    customer
    .join(address, "address_id")
    .join(city, "city_id")
    .groupBy("city_id", "city")
    .agg(
        F.sum(F.when(F.col("active") == 1, 1).otherwise(0)).alias("active_customers"),
        F.sum(F.when(F.col("active") == 0, 1).otherwise(0)).alias("inactive_customers")
    )
    .orderBy(F.desc("inactive_customers"))
)

print("Запрос 6: Города с активными и неактивными клиентами")
task6.show(10)

Запрос 6: Города с активными и неактивными клиентами
+-------+--------------------+----------------+------------------+
|city_id|                city|active_customers|inactive_customers|
+-------+--------------------+----------------+------------------+
|    452|San Juan Bautista...|             383|                18|
|    495|     Southend-on-Sea|               0|                 1|
|     57|             Bat Yam|               0|                 1|
|    407|           Pingxiang|               0|                 1|
|    125|       Coatzacoalcos|               0|                 1|
|    283|          Kumbakonam|               0|                 1|
|    111|    Charlotte Amalie|               0|                 1|
|    356|           Najafabad|               0|                 1|
|    554|            Uluberia|               0|                 1|
|    578|            Xiangfan|               0|                 1|
+-------+--------------------+----------------+------------------+
only show

In [15]:
rental_duration = (
    rental.filter(F.col("return_date").isNotNull())
    .withColumn(
        "rental_hours",
        (F.unix_timestamp("return_date") - F.unix_timestamp("rental_date")) / 3600
    )
    .join(customer, "customer_id")
    .join(address, "address_id")
    .join(city, "city_id")
    .join(inventory, "inventory_id")
    .join(film_category, "film_id")
    .join(category, "category_id")
    .filter(
        F.lower(F.col("name")).like("a%") | F.col("city").like("%-%")
    )
    .groupBy("city_id", "city", "name")
    .agg(F.sum("rental_hours").alias("total_rental_duration"))
)

window_spec = Window.partitionBy("city_id").orderBy(F.desc("total_rental_duration"))

task7 = (
    rental_duration
    .withColumn("rnk", F.rank().over(window_spec))  
    .filter(F.col("rnk") == 1)
    .select("city", F.col("name").alias("category_name"), "total_rental_duration")
)

print('Запрос 7: Категории фильмов с наибольшим общим количеством прокатов')
task7.show(5)

Запрос 7: Категории фильмов с наибольшим общим количеством прокатов
+------------------+-------------+---------------------+
|              city|category_name|total_rental_duration|
+------------------+-------------+---------------------+
|A Corua (La Corua)|    Animation|   1156.6847222222223|
|              Abha|       Action|   2067.7866666666664|
|         Abu Dhabi|    Animation|   1749.9877777777779|
|              Acua|       Action|   1518.1291666666666|
|             Adana|    Animation|   1879.4594444444447|
+------------------+-------------+---------------------+
only showing top 5 rows
